# ChurnShield — 01. Exploratory Data Analysis

Goal: identify customer groups associated with higher churn risk using the IBM Telco Churn schema. For the final internship submission, run this notebook after downloading the official Kaggle CSV.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve()))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.preprocessing import load_raw, clean_data, encode_target

df_raw = load_raw()
df_raw.head()

## 1. Dataset inspection

In [ ]:
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
display(df_raw.dtypes.to_frame("dtype"))
display(df_raw.describe(include="all").T.head(25))

## 2. Missing values and duplicates

In [ ]:
print("Missing values:")
display(df_raw.isna().sum().sort_values(ascending=False).to_frame("missing"))
print("Duplicate customer IDs:", df_raw.duplicated("customerID").sum())
total_num = pd.to_numeric(df_raw["TotalCharges"], errors="coerce")
print("Non-numeric TotalCharges rows:", total_num.isna().sum())

## 3. Clean data

In [ ]:
df = encode_target(clean_data(df_raw))
print("Cleaned shape:", df.shape)
display(df.head())

## 4. Churn distribution

In [ ]:
churn_pct = df["Churn"].value_counts(normalize=True).sort_index()*100
print(churn_pct)
counts = df["Churn"].value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(counts.index.astype(str), counts.values)
plt.title("Churn Distribution")
plt.xlabel("Churn (0=No, 1=Yes)")
plt.ylabel("Customers")
plt.tight_layout()
plt.show()


## 5. Churn by key categorical features

In [ ]:
def churn_rate_table(column):
    return df.groupby(column)["Churn"].mean().sort_values(ascending=False).mul(100).round(2)

for col in ["Contract", "InternetService", "PaymentMethod", "TechSupport", "OnlineSecurity"]:
    print("\n", col)
    display(churn_rate_table(col).to_frame("churn_rate_pct"))

## 6. Numerical relationships

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.boxplot(column="tenure", by="Churn", ax=axes[0])
axes[0].set_title("Tenure by Churn")
axes[0].set_xlabel("Churn (0=No, 1=Yes)")
axes[0].set_ylabel("Tenure (months)")
df.boxplot(column="MonthlyCharges", by="Churn", ax=axes[1])
axes[1].set_title("Monthly Charges by Churn")
axes[1].set_xlabel("Churn (0=No, 1=Yes)")
axes[1].set_ylabel("Monthly Charges")
fig.suptitle("")
plt.tight_layout()
plt.show()


## 7. Correlation matrix

In [ ]:
num_df = df[["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]].copy()
corr = num_df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr.values, vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
ax.set_title("Numeric Feature Correlation Matrix")
fig.colorbar(im, ax=ax, shrink=0.8, label="Correlation")
plt.tight_layout()
plt.show()


## EDA conclusion

Use the tables and charts above to identify observed churn associations. Keep the language associative rather than causal.